# T8 - Post-Image Processing (standalone)
Input: B-mode `(depth x lines)` in [0, 1] plus its `x_lines` and `z_axis` (metres). Steps: scan conversion to square pixels, then guided-filter despeckle. This notebook builds a synthetic speckled B-mode with a dark cyst so it runs alone; set `BMODE_FILE` to use a real `06_bmode_outputs.npz`.

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
from post_processing_stage import scan_convert, postprocess, speckle_index, to_uint8

In [ ]:
BMODE_FILE = None     # e.g. ROOT/'integration/03_full_pipeline/outputs/<run>/06_bmode_outputs.npz'
if BMODE_FILE:
    d = np.load(BMODE_FILE); bmode, z_axis, x_lines = d['bmode'], d['z_axis'], d['x_lines']
else:
    from scipy.ndimage import gaussian_filter
    rng = np.random.default_rng(0); nz, nx = 600, 127
    z_axis = np.linspace(0, 30e-3, nz); x_lines = np.linspace(-9.8e-3, 9.8e-3, nx)
    sp = np.abs(gaussian_filter(rng.normal(size=(nz, nx)), (3, 1)) + 1j*gaussian_filter(rng.normal(size=(nz, nx)), (3, 1)))
    Z, X = np.meshgrid(z_axis, x_lines, indexing='ij')
    sp[np.hypot(X - 2e-3, Z - 15e-3) < 3e-3] *= 0.08           # dark cyst
    db = 20*np.log10(sp/sp.max()); bmode = (np.clip(db, -50, 0) + 50)/50
print('bmode', bmode.shape, '| z', z_axis[[0, -1]]*1e3, 'mm | x', x_lines[[0, -1]]*1e3, 'mm')

## 1. Scan conversion
The beamformed grid has different spacing in depth and lateral direction; this resamples it to square display pixels.

In [ ]:
image, x_img, z_img = scan_convert(bmode, x_lines, z_axis)
print('image', image.shape)
ext = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
fig, ax = plt.subplots(1, 2, figsize=(10, 6))
ax[0].imshow(bmode, cmap='gray', aspect='auto'); ax[0].set_title('B-mode (depth x lines)')
ax[1].imshow(image, cmap='gray', extent=ext); ax[1].set_title('scan converted'); plt.show()

## 2. Despeckle (guided filter)
Lower speckle index = smoother.

In [ ]:
variants = {'none': postprocess(image, 'none'), 'guided r=4': postprocess(image, 'guided', r=4, eps=0.001), 'guided r=8': postprocess(image, 'guided', r=8, eps=0.001)}
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, (k, v) in zip(ax, variants.items()):
    a.imshow(v, cmap='gray', extent=ext, vmin=0, vmax=1); a.set_title(f'{k}  (speckle {speckle_index(to_uint8(v)):.3f})')
plt.tight_layout(); plt.show()

In [ ]:
final = variants['guided r=4']; print('final image', final.shape, final.min(), final.max())